# DataForSEO SERP Collection Starter Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lazarinastoy/mlforseo-code/blob/main/dataforseo-serp-collection/dataforseo_serp_collection.ipynb)

By **Lazarina Stoy, [MLforSEO](https://www.mlforseo.com/)**. Free, MIT licence.

Most SERP-based machine learning work in SEO starts the same way: you need the search results for a list of keywords, in a table you can actually work with. This notebook is that first step. It:

1. **Cleans your keyword list** (no paying twice for the same SERP).
2. **Estimates the cost** of the run before you spend anything.
3. **Collects Google organic SERPs** through the DataForSEO SERP API v3, either *Live Advanced* (instant) or *Standard* (`task_post` + `task_get`, cheaper and slower).
4. **Logs what every call cost** and stops at a spend cap you set.
5. **Flattens the JSON** into tidy tables: organic results, SERP features, People Also Ask, AI Overview citations and related searches.
6. **Exports** CSVs and one Sheets-ready Excel workbook.

**What you need:** a DataForSEO account with API access (login + API password from [app.dataforseo.com/api-access](https://app.dataforseo.com/api-access)). New accounts get a small free trial credit. You don't need anything to try it: with `USE_SAMPLE = True` it runs on three real SERPs saved in `sample_data/` (US, English, desktop, pulled 24 October 2026).

**What it costs (October 2026 prices, check the [pricing page](https://dataforseo.com/pricing/serp/google-organic-serp-api)):** about $0.002 per keyword in Live mode and $0.0006 in Standard mode for the top 10 results, plus an AI Overview surcharge that is refunded when there's nothing extra to load. 100 keywords in Live mode is roughly $0.20 to $0.40.

## 1. Setup

In Colab this installs the two small libraries it needs and downloads the helper file and sample data from GitHub. Locally, run it from inside the `dataforseo-serp-collection/` folder.

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_RAW = "https://raw.githubusercontent.com/lazarinastoy/mlforseo-code/main/dataforseo-serp-collection"
FILES = ["dataforseo_serp.py", "sample_data/keywords.csv", "sample_data/what-is-entity-seo.json",
         "sample_data/best-running-shoes.json", "sample_data/screaming-frog-download.json"]

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "requests", "openpyxl"], check=True)
for f in FILES:  # only downloads what isn't already next to the notebook
    if not Path(f).exists():
        Path(f).parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["wget", "-q", "-O", f, f"{REPO_RAW}/{f}"], check=True)

import warnings
warnings.filterwarnings("ignore", message=".*OpenSSL.*")  # harmless urllib3 notice on some Macs
import pandas as pd
import dataforseo_serp as dfs
pd.set_option("display.max_colwidth", 70)
print("Ready.")

Ready.


## 2. Settings

Leave `USE_SAMPLE = True` for a free dry run. Set it to `False` to use your own account.

* `MODE`: `"live"` returns each SERP in a few seconds (one keyword per call). `"standard"` queues up to 100 keywords per call and you collect them a few minutes later, at under a third of the price. Use Live for small lists and testing, Standard for anything you'd run every week.
* `LOCATION_CODE` / `LANGUAGE_CODE`: 2840 = United States, 2826 = United Kingdom ([full list](https://docs.dataforseo.com/v3/serp/google/locations/)).
* `DEPTH`: how many results. You're billed per page of 10, so 10 is the cheapest.
* `LOAD_ASYNC_AI_OVERVIEW`: many AI Overviews load after the page does. Without this you only get the ones Google had already rendered, so you'd undercount them. The surcharge is refunded when there's no asynchronous AI Overview.
* `PAA_CLICKS`: 0 to 4. Each click expands People Also Ask for more questions ($0.00015 per click).
* `MAX_SPEND_USD`: the client refuses to make another paid call once it has spent this much.

In [2]:
USE_SAMPLE = True            # False = call the real API with your credentials

MODE = "live"                # "live" or "standard"
LOCATION_CODE = 2840         # United States
LANGUAGE_CODE = "en"
DEVICE = "desktop"           # or "mobile"
DEPTH = 10                   # results per SERP (billed per page of 10)
LOAD_ASYNC_AI_OVERVIEW = True
PAA_CLICKS = 0               # 0-4
PRIORITY = 1                 # standard mode only: 1 normal, 2 high (costs more)
MAX_SPEND_USD = 0.10         # hard stop for the whole run
MAX_KEYWORDS = 200           # a second guardrail against pasting a huge list by mistake

## 3. Your keywords

Paste them into `KEYWORDS`, or point `KEYWORDS_CSV` at a CSV with a `keyword` column (in Colab, upload it with the folder icon on the left first). The sample file has a deliberate duplicate ("Best Running Shoes ") to show the clean-up.

In [3]:
KEYWORDS_CSV = "sample_data/keywords.csv"
KEYWORDS = []                # or e.g. ["what is entity seo", "entity seo tools"]

raw = KEYWORDS or pd.read_csv(KEYWORDS_CSV)["keyword"].tolist()
keywords = dfs.clean_keywords(raw)
print(f"{len(raw)} keywords in, {len(keywords)} after removing duplicates and blanks")
if len(keywords) > MAX_KEYWORDS:
    raise ValueError(f"{len(keywords)} keywords is over MAX_KEYWORDS ({MAX_KEYWORDS}). Raise it if you meant it.")
keywords

4 keywords in, 3 after removing duplicates and blanks


['what is entity seo', 'best running shoes', 'screaming frog download']

## 4. Estimate the cost first

This is a worst-case figure: it assumes every keyword is charged the AI Overview surcharge and every PAA click happens. The real cost is logged from DataForSEO's own `cost` field after each call.

In [4]:
est = dfs.estimate_cost(len(keywords), mode=MODE, depth=DEPTH,
                        load_async_ai_overview=LOAD_ASYNC_AI_OVERVIEW, paa_clicks=PAA_CLICKS, priority=PRIORITY)
print(pd.Series(est).to_string())
if not USE_SAMPLE and est["usd_total_max"] > MAX_SPEND_USD:
    print(f"\nHeads up: the estimate is above MAX_SPEND_USD (${MAX_SPEND_USD}). "
          "The run will stop when the cap is reached. Raise the cap or trim the list.")

mode                    live
keywords                   3
pages_per_keyword          1
usd_per_keyword_max    0.004
usd_total_max          0.012


## 5. Connect to DataForSEO

In Colab, add two secrets with the key icon in the left sidebar: `DATAFORSEO_LOGIN` and `DATAFORSEO_PASSWORD` (the **API** password, not your dashboard password), and give this notebook access. If they aren't there, you'll get a hidden prompt. Your credentials are never written to the notebook.

In [5]:
if USE_SAMPLE:
    client = dfs.SampleClient("sample_data")
    print("Sample mode: replaying saved API responses for", client.available_keywords())
else:
    client = dfs.DataForSEOClient(*dfs.get_credentials(), max_spend_usd=MAX_SPEND_USD)
    print("Connected. Spend cap: $", MAX_SPEND_USD)

Sample mode: replaying saved API responses for ['best running shoes', 'screaming frog download', 'what is entity seo']


## 6. What one API call looks like

Under the hood, a Live Advanced request is a single POST with HTTP Basic auth and a JSON list holding one task:

```python
import requests
r = requests.post(
    "https://api.dataforseo.com/v3/serp/google/organic/live/advanced",
    auth=(login, password),
    json=[{"keyword": "what is entity seo", "location_code": 2840, "language_code": "en",
           "device": "desktop", "depth": 10, "load_async_ai_overview": True}],
)
data = r.json()   # data["cost"], data["tasks"][0]["result"][0]["items"]
```

Every element on the page comes back in `items`, each with a `type` (`organic`, `ai_overview`, `people_also_ask`, `video`, `related_searches` and many more). Here is the shape of the first keyword:

In [6]:
responses = client.collect(keywords[:1], mode=MODE, location_code=LOCATION_CODE, language_code=LANGUAGE_CODE,
                           device=DEVICE, depth=DEPTH, load_async_ai_overview=LOAD_ASYNC_AI_OVERVIEW,
                           paa_clicks=PAA_CLICKS)
first = dfs.results_from_responses(responses)[0]
print("keyword:   ", first["keyword"])
# the POST is what is billed (in standard mode the task_get that fetched the result is free)
print("cost (USD):", client.cost_log[-1]["cost_usd"] if client.cost_log else 0)
print("item types:", first["item_types"])
pd.DataFrame([{"rank_absolute": i.get("rank_absolute"), "type": i["type"],
               "domain_or_title": i.get("domain") or i.get("title") or ""} for i in first["items"]])

keyword:    what is entity seo
cost (USD): 0.002
item types: ['ai_overview', 'organic', 'people_also_ask', 'related_searches']


,rank_absolute,type,domain_or_title
0,1,ai_overview,
1,2,organic,inlinks.com
2,3,people_also_ask,
3,4,organic,neilpatel.com
4,5,organic,searchengineland.com
5,6,organic,ahrefs.com
6,7,organic,www.schemaapp.com
7,8,organic,www.youtube.com
8,9,organic,www.clearscope.io
9,10,organic,www.found.co.uk


## 7. Collect the whole list

In Live mode each keyword is one call, with a short pause between calls. In Standard mode the notebook posts the tasks in batches of 100, then polls `task_get/advanced` (free) until they're ready, usually within a few minutes at normal priority. Either way you end up with the same JSON.

In [7]:
params = dict(location_code=LOCATION_CODE, language_code=LANGUAGE_CODE, device=DEVICE, depth=DEPTH,
              load_async_ai_overview=LOAD_ASYNC_AI_OVERVIEW, paa_clicks=PAA_CLICKS)
if MODE == "standard":
    params["priority"] = PRIORITY
# keyword 1 was already collected in step 6, so only the rest are fetched (no paying twice)
responses = responses[:1] + client.collect(keywords[1:], mode=MODE, **params)

cost_log = pd.DataFrame(client.cost_log)
print(f"{len(responses)} SERPs collected. Spent this run: ${client.spent:.4f}"
      + ("  (sample mode: replaying saved data costs nothing)" if USE_SAMPLE else ""))
cost_log

3 SERPs collected. Spent this run: $0.0000  (sample mode: replaying saved data costs nothing)


,endpoint,label,cost_usd,spent_so_far_usd
0,sample replay,what is entity seo,0.002,0.0
1,sample replay,best running shoes,0.002,0.0
2,sample replay,screaming frog download,0.002,0.0


## 8. Flatten to tidy tables

`dfs.flatten()` walks every item and writes one row per thing you'd want to analyse. Domains lose their `www.` so the same site counts once.

In [8]:
tables = dfs.flatten(responses)
for name, df in tables.items():
    print(f"{name:24s} {len(df):4d} rows")
tables["organic"][["keyword", "rank_group", "domain", "title"]].head(12)

organic                    26 rows
serp_features              36 rows
paa                        12 rows
ai_overview_citations      19 rows
related_searches           24 rows


,keyword,rank_group,domain,title
0,what is entity seo,1,inlinks.com,Entity SEO: the Guide to Understanding
1,what is entity seo,2,neilpatel.com,Entity-Based SEO: What Is Entity SEO & Why Does It Matter?
2,what is entity seo,3,searchengineland.com,Entity SEO: The definitive guide
3,what is entity seo,4,ahrefs.com,What is Entity-Based SEO?
4,what is entity seo,5,schemaapp.com,What is Entity SEO and How Do I Implement It?
5,what is entity seo,6,youtube.com,Entity SEO: Connect the Dots and Rank Higher
6,what is entity seo,7,clearscope.io,What Is an Entity and Why Does It Matter for SEO
7,what is entity seo,8,found.co.uk,Entity SEO Guide: What is Entity Based SEO | Found
8,what is entity seo,9,seoforjournalism.com,What is entity SEO?
9,best running shoes,1,weartesters.com,Best Long Distance Running Shoes


### SERP features by keyword

One row per keyword, one column per SERP element type (1 = present). This is the input for SERP-feature intent classification and for spotting which keywords show an AI Overview.

In [9]:
dfs.feature_matrix(tables["serp_features"])

item_type,ai_overview,organic,people_also_ask,related_searches,video
keyword,,,,,
best running shoes,1,1,1,1,0
screaming frog download,1,1,1,1,1
what is entity seo,1,1,1,1,0


### People Also Ask and AI Overview citations

AI Overview citations come from the `references` arrays inside the `ai_overview` item. DataForSEO lists them per paragraph and again in the source panel beside the answer; the helper keeps each URL once, in order of first appearance.

In [10]:
display(tables["paa"][["keyword", "paa_position", "question"]])
tables["ai_overview_citations"][["keyword", "citation_order", "domain", "aio_section"]].head(15)

,keyword,paa_position,question
0,what is entity seo,1,What are four types of SEO?
1,what is entity seo,2,Is SEO dead now with AI?
2,what is entity seo,3,What is an example of an entity?
3,what is entity seo,4,What does SEO mean in simple terms?
4,best running shoes,1,How do I clean my running shoes?
5,best running shoes,2,Is Asics or Nike better?
6,best running shoes,3,How many miles should running shoes last?
7,best running shoes,4,Can running shoes be used for walking?
8,screaming frog download,1,How to get Screaming Frog for free?
9,screaming frog download,2,How much does a Screaming Frog license cost?


,keyword,citation_order,domain,aio_section
0,what is entity seo,1,neilpatel.com,
1,what is entity seo,2,demandspring.com,
2,what is entity seo,3,inlinks.com,
3,what is entity seo,4,schemaapp.com,
4,what is entity seo,5,ahrefs.com,
5,what is entity seo,6,youtube.com,Core Concepts: Keywords vs. Entities
6,what is entity seo,7,seoforjournalism.com,Core Concepts: Keywords vs. Entities
7,what is entity seo,8,studiohawk.com.au,Core Concepts: Keywords vs. Entities
8,what is entity seo,9,schemaapp.com,Key Tactics for Implementation
9,what is entity seo,10,youtube.com,Key Tactics for Implementation


## 9. Export

CSVs for each table, plus `serp_collection.xlsx` with one tab per table. In Google Sheets use *File → Import → Upload* to bring the workbook in. In Colab, download from the folder icon on the left (`output/`).

In [11]:
for p in dfs.export(tables, "output", xlsx_name="serp_collection.xlsx"):
    print(p)
cost_log.to_csv("output/cost_log.csv", index=False)

output/organic.csv
output/serp_features.csv
output/paa.csv
output/ai_overview_citations.csv
output/related_searches.csv
output/serp_collection.xlsx


## How to read the output

* **organic.csv**: `rank_group` is the position among organic results only; `rank_absolute` counts every element on the page (an AI Overview at the top pushes the first organic result to `rank_absolute` 2).
* **serp_features.csv**: one row per SERP element. Pivot it (as above) to compare keywords.
* **paa.csv**: People Also Ask questions in display order. Add `PAA_CLICKS` for more.
* **ai_overview_citations.csv**: every URL the AI Overview cited, with the section it supported.
* **cost_log.csv**: what each call actually cost.

## Limits and honest caveats

* A SERP is one snapshot from one location, language and device at one moment. AI Overviews in particular change between pulls; track over time before drawing conclusions.
* Prices and limits are DataForSEO's and change. The estimate uses October 2026 list prices; the cost log uses what the API really charged.
* Live mode takes one task per call. For lists of hundreds or thousands, use Standard mode (cheaper, and up to 100 tasks per POST).
* The sample data is trimmed (no pixel positions, markdown or long snippets) to keep the files small; the structure is otherwise exactly what the API returns.

## What to do next

* Track AI Overview presence and citation share for your own domain with the companion notebook, [AI Overview Tracker](https://github.com/lazarinastoy/mlforseo-code/tree/main/ai-overview-tracker), which builds on this one.
* Take the SERP data further (SERP-feature intent, n-gram gaps, SERP-overlap clustering) with [`serp-analysis-machine-learning/`](https://github.com/lazarinastoy/mlforseo-code/tree/main/serp-analysis-machine-learning).
* The full SERP collection lab, with keyword clustering and intent work built on top of it, is part of the [Semantic ML-enabled Keyword Research course](https://www.mlforseo.com/courses/semantic-ml-enabled-keyword-research/).